# Kvasir-SEG DeepLabV3-ResNet50 Baseline

Runs a DeepLab-based baseline with torchvision and writes standardized artifacts under `out/`.

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import os
import json
from pathlib import Path

import torch

from utils.segmentation_common import (
    find_kvasir_seg_root,
    load_metadata,
    TrainConfig,
    build_deeplab_resnet50,
    train_and_evaluate,
)

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
SPLIT_HASH_TXT = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'split_hash.txt'
OUT_DIR = ROOT / '1_classic_seg_baselines' / 'out' / 'deeplabv3plus_resnet50_baseline'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv('SEED', '42'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '4'))
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '2'))
EPOCHS = int(os.getenv('EPOCHS', '6'))
LR = float(os.getenv('LR', '3e-4'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-4'))
IMAGE_SIZE = int(os.getenv('IMAGE_SIZE', '352'))
THRESHOLD = float(os.getenv('THRESHOLD', '0.5'))

MAX_TRAIN = int(os.getenv('MAX_TRAIN_SAMPLES', '0')) or None
MAX_VAL = int(os.getenv('MAX_VAL_SAMPLES', '0')) or None
MAX_TEST = int(os.getenv('MAX_TEST_SAMPLES', '0')) or None

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('CUDA available:', torch.cuda.is_available())


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
OUT_DIR: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/1_classic_seg_baselines/out/deeplabv3plus_resnet50_baseline
CUDA available: True


In [3]:

meta_df = load_metadata(META_CSV)
split_hash = SPLIT_HASH_TXT.read_text().strip() if SPLIT_HASH_TXT.exists() else None

cfg = TrainConfig(
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    threshold=THRESHOLD,
    compute_hd95=False,
    save_pred_masks=True,
    pred_mask_limit=200,
)

model = build_deeplab_resnet50(num_classes=1)


In [4]:

results = train_and_evaluate(
    model=model,
    model_name='deeplabv3_resnet50',
    root=ROOT,
    out_dir=OUT_DIR,
    cfg=cfg,
    metadata_df=meta_df,
    split_hash=split_hash,
    max_train=MAX_TRAIN,
    max_val=MAX_VAL,
    max_test=MAX_TEST,
)

print(json.dumps(results, indent=2))


Epoch 1/6 train_loss=0.5051 val_loss=0.5444 val_dice=0.3669
Epoch 2/6 train_loss=0.4387 val_loss=0.4638 val_dice=0.4862
Epoch 3/6 train_loss=0.4008 val_loss=0.4099 val_dice=0.5515
Epoch 4/6 train_loss=0.3607 val_loss=0.4153 val_dice=0.5744
Epoch 5/6 train_loss=0.3218 val_loss=0.3572 val_dice=0.6287
Epoch 6/6 train_loss=0.2869 val_loss=0.3898 val_dice=0.5989


/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/utils/segmentation_common.py:1390: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_st

{
  "train": {
    "n": 800,
    "dice_mean": 0.6955176461122385,
    "dice_median": 0.7774379390539462,
    "dice_std": 0.24838480092996632,
    "iou_mean": 0.5812779713686338,
    "iou_median": 0.635908794703425,
    "iou_std": 0.2571669076863802,
    "precision_mean": 0.7560728253053195,
    "precision_median": 0.8754773958845299,
    "precision_std": 0.28196909003060766,
    "recall_mean": 0.7303488984787798,
    "recall_median": 0.8328377707846981,
    "recall_std": 0.2733828304618628,
    "f1_mean": 0.6955176416722282,
    "f1_median": 0.7774379341877173,
    "f1_std": 0.24838480005739336,
    "specificity_mean": 0.9712837761117669,
    "specificity_median": 0.9870280355802155,
    "specificity_std": 0.03950271251876232,
    "loss": 0.2950394606590271
  },
  "val": {
    "n": 100,
    "dice_mean": 0.6286982322097383,
    "dice_median": 0.727850132346249,
    "dice_std": 0.2943130803730366,
    "iou_mean": 0.5209685642502235,
    "iou_median": 0.5721440441836976,
    "iou_std": 0.